In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 6: LLM Fine-Tuning for Domain Tasks (LoRA/QLoRA)

### Business Context
Fine-tune an LLM (e.g. LLaMA-3) for bid recommendations in paid search. Given campaign features (CTR, ROAS, budget utilization), the model outputs structured bid adjustment recommendations.

### LoRA (Low-Rank Adaptation)
Instead of updating all W (billions of params), inject trainable low-rank matrices:
```
W' = W_0 + B·A    where B ∈ R^(d×r), A ∈ R^(r×k), r << min(d,k)
```
Only B and A are trained (~0.1-1% of total parameters). W_0 is frozen.

**Rank selection guide:**
- r=4-8: style transfer, formatting tasks
- r=16-32: domain adaptation (typical for bid recs)
- r=64-128: complex reasoning, near full fine-tuning quality

**Alpha scaling:** Set α = 2r. This makes the effective learning rate scale-independent of r.

### QLoRA
Quantize the base model to 4-bit (NF4 quantization), apply LoRA on top. Fits a 65B model on a single 48GB GPU. Training remains in bfloat16 through compute.

### SFT vs DPO
| | SFT (Supervised Fine-Tuning) | DPO (Direct Preference Optimization) |
|---|---|---|
| Data | (prompt, completion) pairs | (prompt, chosen, rejected) triples |
| Process | Standard cross-entropy on completions | Directly optimizes preference without reward model |
| Complexity | Low | Medium (needs preference data) |
| Use when | Clear correct answer exists | Quality requires preference ranking |

In [ ]:
# --- LoRA weight update simulation ---

class LoRALayer(nn.Module):
    """LoRA adapter for a single linear layer."""
    def __init__(self, base_weight, rank=16, alpha=32, dropout=0.05):
        super().__init__()
        d_out, d_in = base_weight.shape
        self.base_weight = nn.Parameter(base_weight, requires_grad=False)  # frozen
        self.A = nn.Parameter(torch.randn(rank, d_in) * 0.01)   # initialized small
        self.B = nn.Parameter(torch.zeros(d_out, rank))          # initialized to 0
        self.scale = alpha / rank                                 # alpha / r scaling
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # W' = W_0 + scale * B @ A
        base_out = F.linear(x, self.base_weight)
        lora_out = self.scale * F.linear(self.dropout(x), self.A.T @ self.B.T)  # B @ A @ x
        return base_out + F.linear(x, self.scale * (self.B @ self.A))

# Inspect parameter counts
d_model, r = 4096, 16
base_params = d_model * d_model
lora_params = r * d_model + d_model * r   # A + B
print(f'Base layer params: {base_params:,}')
print(f'LoRA params (r={r}):  {lora_params:,}')
print(f'Reduction: {lora_params/base_params*100:.1f}% of original')

# Demonstrate LoRA forward pass
W0 = torch.randn(256, 256)
lora = LoRALayer(W0, rank=16, alpha=32)
x = torch.randn(8, 256)
out = lora(x)
print(f'\nLoRA output shape: {out.shape}')

# Count trainable vs frozen
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in lora.parameters() if not p.requires_grad)
print(f'Trainable params: {trainable:,}  Frozen: {frozen:,}')

# --- Structured output evaluation (beyond perplexity) ---
def evaluate_bid_recommendations(predictions, ground_truth):
    """
    Evaluate LLM bid recs beyond perplexity.
    predictions: list of dicts with 'bid_change_pct' and 'reasoning'
    ground_truth: list of dicts with 'optimal_bid_change_pct'
    """
    results = {}

    # 1. JSON validity rate
    valid = sum(1 for p in predictions if isinstance(p, dict) and 'bid_change_pct' in p)
    results['json_validity'] = valid / len(predictions)

    # 2. Directional accuracy (correct increase/decrease/hold)
    correct_direction = sum(
        1 for p, g in zip(predictions, ground_truth)
        if isinstance(p, dict) and np.sign(p.get('bid_change_pct', 0)) == np.sign(g['optimal_bid_change_pct'])
    )
    results['directional_accuracy'] = correct_direction / len(predictions)

    # 3. Magnitude error (MAPE)
    mape_vals = [
        abs(p.get('bid_change_pct', 0) - g['optimal_bid_change_pct']) / (abs(g['optimal_bid_change_pct']) + 1e-6)
        for p, g in zip(predictions, ground_truth)
        if isinstance(p, dict)
    ]
    results['mape'] = np.mean(mape_vals) if mape_vals else 1.0

    return results

# Synthetic evaluation
preds = [{'bid_change_pct': np.random.uniform(-0.3, 0.3), 'reasoning': 'x'} for _ in range(50)]
truth = [{'optimal_bid_change_pct': np.random.uniform(-0.3, 0.3)} for _ in range(50)]
metrics = evaluate_bid_recommendations(preds, truth)
print(f'\nEvaluation metrics: {metrics}')

### L6 Interview Q&A

**Q: When should you fine-tune vs. just prompt-engineer?**  
A: Always measure prompt-engineering baseline first. Fine-tune when: (1) task requires domain knowledge not in pretraining, (2) structured output format must be reliable (JSON schema), (3) latency/cost require a smaller model, (4) privacy prevents sending data to external APIs. If prompt engineering achieves 90% of the target metric, fine-tuning may not be worth the infra cost.

**Q: What is catastrophic forgetting and how do you mitigate it?**  
A: When fine-tuning on domain data, the model overwrites weights used for general language understanding, degrading performance on tasks outside the fine-tuning distribution. Mitigations: LoRA (only adapters are trained, base is frozen), data mixing (blend domain data with general text), early stopping on a held-out general benchmark, Elastic Weight Consolidation (penalizes changing weights important to the original task).

**Q: Why is perplexity insufficient for evaluating bid recommendation quality?**  
A: Perplexity measures how well the model predicts the next token — it's a language quality metric, not a task quality metric. A model can have low perplexity while outputting syntactically plausible but directionally wrong bid recommendations. Measure JSON validity, directional accuracy (increase/decrease/hold), magnitude error (MAPE), and simulate P&L impact using the recommendations on held-out campaign data.

### Common Pitfalls
- Using one prompt template at training, a different one at inference — distribution shift
- Not logging the behavioral policy's π_b(a|s) — needed for OPE later
- Setting rank too high without enough data — overfitting risk